# X10D2 — Project One: Group Challenge

## ExoticMeatMarkets.com — Full Dallas Expansion

Over the past several weeks you've built a complete delivery optimization pipeline:
- Geocoding addresses and building road network graphs (X9D1)
- Greedy allocation by margin and TSP routing (X9D2)
- Refactoring into reusable functions and comparing truck scenarios (X10D1)

**Today the training wheels come off.**

ExoticMeatMarkets.com has received orders from **29 Dallas restaurants** — up from 10.
The supplier has also expanded: **5 trucks** are now available to rent,
each with a different cost structure and capacity.
You may also deploy **up to two trucks simultaneously** on a given day.

### Company Headquarters — Dallas Love Field
All routes begin and end at the ExoticMeatMarkets.com distribution hub:

> **8333 George Coker Cir, Dallas, TX** *(Dallas Love Field)*

The Cuy arrives here each morning. Your truck(s) depart from this address,
complete the delivery route, and return. **Route distance must include the
leg from HQ to the first stop and the return leg from the last stop back to HQ.**

### Real-World Variability
On actual delivery days, supply levels and restaurant demand shift constantly.
**Your solution must be dynamic** — it should work correctly for any combination
of supply availability and bid prices, not just the values in `restaurants_full.xlsx`.
Your group will be given a specific supply scenario at the start of X11D2;
your notebook should be ready to re-run with new inputs in minutes.

Your group's goal: **build a solution that dynamically identifies the most profitable
1- or 2-truck configuration given any supply level, demand, and bid prices.**

---

## Project Deliverables

| Item | Due |
|------|-----|
| Group work session (in class) | Today, X10D2 — last 30 minutes |
| Full work day | X11D1 (Tuesday) |
| **Final notebook submitted to GitHub + Canvas** | **X11D2 (Thursday), start of class** |

**One submission per group.** Submit a link to your GitHub repository on Canvas.

---

## The Data Files

You have two new files for this project:

### `restaurants_full.xlsx`
29 restaurants across Dallas with four columns:
- `RESTAURANT` — name
- `ADDRESS` — street address (Dallas, TX)
- `ENTREES_REQUESTED` — how many entrees each restaurant ordered
- `ENTREE_BID_PRICE` — what that restaurant is willing to pay per entree

**Bid prices range from $35 to $95.** Your supply cost remains **$28.00 per entree**.
Margins are tighter than your X9 set — restaurant selection and routing decisions matter more.

⚠️ **These values represent one possible day.** On delivery day (X11D2) your instructor
will announce the actual supply available and may adjust bid prices. Your notebook
must accept `DAILY_SUPPLY_CAP` as a variable — never hardcode it.

### `trucks.csv`
Five trucks available to rent with four columns:
- `truck_name`
- `fixed_cost` — flat daily fee regardless of distance
- `per_mile_cost` — variable cost per mile driven
- `capacity` — maximum entrees the truck can carry

**No single truck dominates.** A truck with low fixed cost may have high per-mile cost.
A high-capacity truck costs more to deploy. You must run scenarios to find the winner.


## Constraints and Rules

### Starting Point — Company HQ
Every route starts **and ends** at:
> **8333 George Coker Cir, Dallas, TX** (Dallas Love Field)

Geocode this address exactly once. Snap it to its nearest OSMnx node.
The first leg of every route is HQ → first restaurant stop.
The last leg is final restaurant stop → HQ.
Total miles must include both of these legs.

### Supply Cap — Variable by Day
ExoticMeatMarkets.com can supply up to **165 entrees** in the default scenario,
across both trucks combined if you deploy two. **However, the actual supply
available on delivery day will be announced by your instructor at the start of X11D2
and may differ.** Your notebook must treat `DAILY_SUPPLY_CAP` as a variable
that can be changed in a single cell without breaking anything downstream.

### Single-Truck Rules
- You pick **one truck** from `trucks.csv`
- That truck's `capacity` caps how many entrees it can physically carry
- Effective cap = `min(DAILY_SUPPLY_CAP, truck.capacity)`
- Route the truck through your selected restaurants (greedy TSP, as before)
- Compute the full P&L

### Two-Truck Rules (optional — but potentially more profitable)
- You pick **two different trucks** from `trucks.csv`
- You **split the 29 restaurants** between the two trucks however you like
- Each truck gets its own allocation and its own route
- Each truck pays its own fixed + variable costs
- **Combined entrees across both trucks ≤ 165** (the daily supply cap still applies)
- **Each truck must serve at least 3 restaurants** (a one-stop delivery doesn't justify the truck)
- Net profit = Truck 1 net profit + Truck 2 net profit

### General Rules
- A restaurant cannot be served by both trucks
- You cannot serve more entrees than a restaurant requested
- Partial fills are allowed (serve fewer entrees than requested if supply runs out)
- All routes use real Dallas road distances (OSMnx graph from X9D1/X10D1)


## Scoring Rubric

| Category | Points | Description |
|----------|--------|-------------|
| **Correctness** | 25 | P&L math is right; HQ-anchored routes; constraints respected; assertions pass |
| **Code Quality** | 20 | Functions used appropriately; DRY (don't repeat your code); readable; docstrings |
| **Recommendation** | 15 | Produces clear recommendations with numbers to back it up |
| **Validation** | 10 | At least 5 assert-based checks covering key constraints and math |
| **Creativity** | 30 | A creative approach that distinguishes your solution from others (didn't just run it through Claude) |
| **Total** | 100 | |

### Correctness details
Your notebook must produce a **final P&L table** showing:
- Revenue, supply cost, gross profit
- Truck fixed cost, truck variable cost (miles × per_mile_cost)
- Net profit

If deploying two trucks, show the P&L for each truck and a combined summary row.
Miles must include the HQ departure and return legs.

### Scenario Exploration details
Your notebook must show **at least three distinct scenarios** you evaluated:
- At minimum: each single-truck option you seriously considered
- At least one two-truck scenario
- At least two different supply cap values tested (e.g., 100, 165)
- A comparison table at the end showing net profit for each scenario

### Recommendation
A markdown cell at the end of your notebook titled **"Our Recommendation"** that:
1. States which truck(s) you recommend and why
2. Cites specific numbers (net profit, miles driven, key restaurants served/skipped)
3. Acknowledges any trade-offs or assumptions in your approach
4. Explains how your recommendation would change at different supply levels

### Creativity details
There is no single formula here — this category rewards groups that go beyond the
scaffold and bring something original. Examples of what earns creativity points:
- A novel restaurant-split strategy for two trucks with a clear rationale
- A function that automatically selects the best 1-vs-2-truck configuration
  given any supply level (no manual testing required)
- A visualization that makes the trade-offs immediately obvious
- A sensitivity analysis showing at which supply level the optimal truck changes
- Any other approach that makes your solution more useful on a real delivery day

The best solutions will be ones your instructor could actually hand to the delivery
driver on the morning of X11D2 and get a confident answer in seconds.


## Hints and Suggested Approach

### Start from X10D1
Your `allocate_supply()`, `build_route()`, and `calculate_pnl()` functions from X10D1
are already the right building blocks. Copy them into your project notebook.

The main changes you'll need:
1. Load `restaurants_full.xlsx` instead of `restaurants_x9.xlsx`
2. Load truck configs from `trucks.csv` instead of hardcoded dicts
3. Add HQ as a fixed start/end node to `build_route()`
4. Make `DAILY_SUPPLY_CAP` easy to change — set it once at the top of your notebook
5. Loop over truck options to compare scenarios; consider automating the selection

### Geocoding HQ
Add the HQ address to your geocoding step as a special row, or geocode it separately.
Snap it to its OSMnx node just like a restaurant. Then modify `build_route()` so
it always starts from the HQ node and returns to it at the end.

```python
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'
# Geocode once and store the node:
# hq_lat, hq_lon = geocode_address(HQ_ADDRESS)
# HQ_NODE = ox.nearest_nodes(G, hq_lon, hq_lat)
```

### Dynamic supply cap
Set this once at the top of your notebook and never reference a literal number again:
```python
DAILY_SUPPLY_CAP = 165   # ← change this one line on delivery day
```
Every downstream function call should pass `DAILY_SUPPLY_CAP` as a variable.
On X11D2 your instructor may announce a different number — you should be able
to update it and re-run the entire notebook in under a minute.

### Loading trucks from CSV
```python
import pandas as pd

trucks_df = pd.read_csv('trucks.csv')
print(trucks_df)

# Convert each row to a dict — same structure as X10D1's TRUCK_A / TRUCK_B
truck_list = trucks_df.to_dict(orient='records')
# truck_list[0] looks like:
# {'truck_name': 'Sparrow', 'fixed_cost': 75.0, 'per_mile_cost': 2.5, 'capacity': 60}
```

### Auto-selecting the best configuration (Creativity opportunity)
Rather than manually inspecting a comparison table, consider writing a function
that accepts `DAILY_SUPPLY_CAP` and returns the best truck (or truck pair)
automatically:
```python
def find_best_configuration(df, truck_list, supply_cap, G, HQ_NODE):
    """
    Evaluate all single-truck scenarios and a selection of two-truck scenarios.
    Return the configuration with the highest net profit.
    """
    # TODO: your group designs this
    pass
```

### Two-truck split strategies to consider
Think about what makes a good split — these are just starting points:
- Geographic split (north Dallas vs south Dallas by latitude)
- Price tier split (high bid-price restaurants on one truck, casual on the other)
- Margin split (top N restaurants by margin on Truck 1, rest on Truck 2)
- Distance-from-HQ split (close restaurants on a small cheap truck, far ones on a larger one)

There is no one right answer. Part of the challenge is figuring out what to test.

### Watch out for
- **Geocoding:** `restaurants_full.xlsx` has 29 addresses plus HQ. Geocoding all 30 from scratch
  will take ~60 seconds. Use the same cache-to-disk pattern from X9D1/X10D1.
  Name your cache file `restaurants_full_geocoded.xlsx`.
- **Graph:** You already have `dallas_drive.graphml`. Reuse it — don't re-download.
- **Column name:** The bid price column is `ENTREE_BID_PRICE` (same as X9). ✓


## Section 1 — Starter Scaffold

The cell below gives you a working starting point.
Complete the `# TODO` sections as a group.

You are **not** required to follow this structure — if your group has a better
approach, use it. But make sure your final notebook covers all rubric categories.


In [ ]:
import os, time
import pandas as pd
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import AntPath
from geopy.geocoders import Nominatim

# ── Constants ─────────────────────────────────────────────────────────────────
SUPPLY_COST_PER_ENTREE = 28.00   # what ExoticMeatMarkets.com charges you
DAILY_SUPPLY_CAP       = 165     # ← UPDATE THIS on delivery day (X11D2)
METERS_PER_MILE        = 1609.34

# Company headquarters — all routes start and end here
HQ_ADDRESS = '8333 George Coker Cir, Dallas, TX'   # Dallas Love Field

print(f'Supply cost per entree: ${SUPPLY_COST_PER_ENTREE:.2f}')
print(f'Daily supply cap:       {DAILY_SUPPLY_CAP} entrees')
print(f'HQ address:             {HQ_ADDRESS}')


### Geocoding HQ
The cell below geocodes HQ and snaps it to the road network.
Run it once — the result is stored in `HQ_NODE` and reused by `build_route()`.
If you already have the cached graph from X10D1, HQ geocoding takes about 2 seconds.


In [ ]:
# ── Geocode HQ (run once; result stored in HQ_NODE after graph loads) ──────────
HQ_GEO_FILE = 'hq_geocoded.csv'

if os.path.exists(HQ_GEO_FILE):
    hq_row  = pd.read_csv(HQ_GEO_FILE).iloc[0]
    HQ_LAT  = float(hq_row['LATITUDE'])
    HQ_LON  = float(hq_row['LONGITUDE'])
    print(f'HQ coords loaded from cache: ({HQ_LAT:.5f}, {HQ_LON:.5f})')
else:
    print('Geocoding HQ address...')
    _geo = Nominatim(user_agent='cuy-delivery-hq')
    time.sleep(2)
    _loc = _geo.geocode(HQ_ADDRESS, timeout=10)
    assert _loc is not None, f'Could not geocode HQ: {HQ_ADDRESS}'
    HQ_LAT, HQ_LON = _loc.latitude, _loc.longitude
    pd.DataFrame([{'ADDRESS': HQ_ADDRESS,
                   'LATITUDE': HQ_LAT,
                   'LONGITUDE': HQ_LON}]).to_csv(HQ_GEO_FILE, index=False)
    print(f'HQ geocoded and cached: ({HQ_LAT:.5f}, {HQ_LON:.5f})')

# HQ_NODE is set after the graph loads (see code_graph cell)


In [ ]:
# ── Load restaurant data ───────────────────────────────────────────────────────
df_raw = pd.read_excel('restaurants_full.xlsx')
print(f'Restaurants loaded: {len(df_raw)}')
print(f'Total entrees demanded: {df_raw["ENTREES_REQUESTED"].sum()}')
print(f'Bid price range: ${df_raw["ENTREE_BID_PRICE"].min()} – ${df_raw["ENTREE_BID_PRICE"].max()}')
display(df_raw.head())

# ── Load truck configs ─────────────────────────────────────────────────────────
trucks_df  = pd.read_csv('trucks.csv')
truck_list = trucks_df.to_dict(orient='records')
print(f'\nTrucks available: {len(truck_list)}')
display(trucks_df)


In [ ]:
# ── Geocode addresses (cached) ─────────────────────────────────────────────────
GEO_FILE = 'restaurants_full_geocoded.xlsx'

if os.path.exists(GEO_FILE):
    geo_df = pd.read_excel(GEO_FILE)
    print(f'Loaded geocoded coords from {GEO_FILE}')
else:
    print('Geocoding 29 addresses — this takes ~60 seconds (one time only)...')
    geolocator = Nominatim(user_agent='cuy-delivery-project1')

    def geocode_address(address):
        try:
            time.sleep(2)
            loc = geolocator.geocode(address, timeout=10)
            return (loc.latitude, loc.longitude) if loc else (None, None)
        except Exception as e:
            print(f'  Error: {address} — {e}')
            return (None, None)

    geo_df = df_raw[['RESTAURANT', 'ADDRESS']].copy()
    geo_df['LATITUDE'], geo_df['LONGITUDE'] = zip(
        *geo_df['ADDRESS'].apply(geocode_address)
    )
    geo_df.to_excel(GEO_FILE, index=False)
    print(f'Geocoding complete — saved to {GEO_FILE}')

df_raw = df_raw.merge(geo_df[['RESTAURANT', 'LATITUDE', 'LONGITUDE']],
                      on='RESTAURANT', how='left')

# Sanity check
bad = df_raw[df_raw['LATITUDE'].isna() | df_raw['LONGITUDE'].isna()]
if not bad.empty:
    print(f'⚠️  {len(bad)} restaurants failed geocoding:')
    display(bad[['RESTAURANT', 'ADDRESS']])
else:
    print(f'✅ All {len(df_raw)} restaurants geocoded successfully')


In [ ]:
# ── Load road network graph (reuse existing file if present) ───────────────────
GRAPH_FILE = 'dallas_drive.graphml'

if os.path.exists(GRAPH_FILE):
    G = ox.load_graphml(GRAPH_FILE)
    print(f'Graph loaded: {len(G.nodes):,} nodes, {len(G.edges):,} edges')
else:
    print('Graph not found — downloading from OpenStreetMap (3–5 min, one time only)...')
    origin = (df_raw['LATITUDE'].mean(), df_raw['LONGITUDE'].mean())
    ox.settings.use_cache = True
    G = ox.graph_from_point(origin, dist=32186, network_type='drive', simplify=True)
    ox.save_graphml(G, filepath=GRAPH_FILE)
    print(f'Graph saved to {GRAPH_FILE}')

# Snap restaurants to nearest graph nodes
df_raw['NODE_ID'] = df_raw.apply(
    lambda r: ox.nearest_nodes(G, r['LONGITUDE'], r['LATITUDE']), axis=1
)

# Snap HQ to its nearest node — used as route start/end
HQ_NODE = ox.nearest_nodes(G, HQ_LON, HQ_LAT)
print(f'HQ snapped to node: {HQ_NODE}')

# Add margin columns
df_raw['NET_MARGIN']      = df_raw['ENTREE_BID_PRICE'] - SUPPLY_COST_PER_ENTREE
df_raw['GROSS_POTENTIAL'] = df_raw['NET_MARGIN'] * df_raw['ENTREES_REQUESTED']

print('\nRestaurants by net margin (descending):')
display(df_raw[['RESTAURANT', 'ENTREES_REQUESTED', 'ENTREE_BID_PRICE',
                'NET_MARGIN', 'GROSS_POTENTIAL']].sort_values('NET_MARGIN', ascending=False))


In [ ]:
# ── Core functions from X10D1 — updated for HQ-anchored routes ───────────────
# Copy your three functions here and make the modifications noted below.

def allocate_supply(df, supply_cap, truck_capacity):
    """Allocate entrees to restaurants by descending net margin.
    Respects both the daily supply cap and the truck's physical capacity.
    Returns a DataFrame of allocated stops.
    """
    # TODO: paste from X10D1
    pass

def build_route(alloc_df, G, hq_node):
    """Build a delivery route starting and ending at HQ (greedy nearest-neighbor TSP).

    Parameters
    ----------
    alloc_df : DataFrame from allocate_supply() — must have NODE_ID column
    G        : OSMnx directed road network graph
    hq_node  : int — graph node for the company HQ (start and end of route)

    Returns
    -------
    DataFrame — stops in visitation order, with DISTANCE_FROM_PREV_M and
                CUMULATIVE_MILES. Distance from HQ to first stop and from
                last stop back to HQ is included in total miles.
    """
    # TODO: paste from X10D1 and add:
    #   - start the route from hq_node (distance HQ → first stop)
    #   - after all stops, add the return leg (last stop → hq_node)
    pass

def calculate_pnl(route_df, truck):
    """Calculate the full P&L for a route + truck combination.
    Reads truck['fixed_cost'] and truck['per_mile_cost'].
    Returns a dict with revenue, supply_cost, gross_profit,
    truck_fixed_cost, truck_variable_cost, net_profit, entrees, miles.
    """
    # TODO: paste from X10D1
    pass

print('Functions defined.')


## Section 2 — Single-Truck Scenarios

Run each truck as a single-truck deployment. Build a comparison table.
Which truck produces the highest net profit when operating alone?


In [ ]:
# ── Run all 5 single-truck scenarios ──────────────────────────────────────────
single_results = []

for truck in truck_list:
    alloc = allocate_supply(df_raw, DAILY_SUPPLY_CAP, truck['capacity'])
    route = build_route(alloc, G, HQ_NODE)
    pnl   = calculate_pnl(route, truck)
    single_results.append({
        'truck':        truck['truck_name'],
        'capacity':     truck['capacity'],
        'fixed_cost':   truck['fixed_cost'],
        'per_mile':     truck['per_mile_cost'],
        'entrees':      pnl['entrees'],
        'miles':        round(pnl['miles'], 2),
        'revenue':      pnl['revenue'],
        'supply_cost':  pnl['supply_cost'],
        'truck_cost':   pnl['truck_fixed_cost'] + pnl['truck_variable_cost'],
        'net_profit':   pnl['net_profit'],
    })

single_df = pd.DataFrame(single_results).sort_values('net_profit', ascending=False)
print('=== Single-Truck Comparison ===')
display(single_df)
print(f'\nBest single-truck option: {single_df.iloc[0]["truck"]} '
      f'(${single_df.iloc[0]["net_profit"]:,.2f})')


## Section 3 — Two-Truck Scenarios (Optional Extension)

If you choose to explore a two-truck deployment, design your restaurant split
strategy here. Each group will approach this differently — document your thinking.

**Key question:** Does splitting the 29 restaurants across two smaller trucks
beat running one large truck?

Think about:
- Which restaurants cluster geographically? (Check the map from X10D1)
- Which truck combination minimizes total fixed cost while preserving capacity?
- Does the supply cap (165 total entrees) change which restaurants you prioritize?


In [ ]:
# ── Two-truck split — design your strategy here ────────────────────────────────
# Example: split by latitude (north Dallas / south Dallas)
# You are free to use any split strategy — just document it.

# TODO: define df_truck1 and df_truck2 as subsets of df_raw
# Rules:
#   - No restaurant appears in both subsets
#   - Each subset has at least 3 restaurants
#   - Combined entrees allocated across both trucks <= DAILY_SUPPLY_CAP

# Example split by latitude:
median_lat = df_raw['LATITUDE'].median()
df_north   = df_raw[df_raw['LATITUDE'] >= median_lat].copy()
df_south   = df_raw[df_raw['LATITUDE'] <  median_lat].copy()

print(f'North restaurants: {len(df_north)}')
print(f'South restaurants: {len(df_south)}')
print('\n--- This is just a starting point ---')
print('Your group should experiment with different splits.')


In [ ]:
# ── Two-truck P&L ──────────────────────────────────────────────────────────────
# TODO: choose your two trucks and calculate combined profit

# Example: Sparrow (north) + Osprey (south)
truck1 = next(t for t in truck_list if t['truck_name'] == 'Sparrow')
truck2 = next(t for t in truck_list if t['truck_name'] == 'Osprey')

# Each truck gets a portion of the daily supply cap
# How you split the 165 cap between trucks is a decision your group must make
CAP_TRUCK1 = 70    # TODO: adjust
CAP_TRUCK2 = 95    # TODO: adjust — must sum to <= 165

assert CAP_TRUCK1 + CAP_TRUCK2 <= DAILY_SUPPLY_CAP, \
    f'Combined supply cap {CAP_TRUCK1 + CAP_TRUCK2} exceeds daily limit {DAILY_SUPPLY_CAP}'

alloc1 = allocate_supply(df_north, CAP_TRUCK1, truck1['capacity'])
route1 = build_route(alloc1, G, HQ_NODE)
pnl1   = calculate_pnl(route1, truck1)

alloc2 = allocate_supply(df_south, CAP_TRUCK2, truck2['capacity'])
route2 = build_route(alloc2, G, HQ_NODE)
pnl2   = calculate_pnl(route2, truck2)

combined_profit = pnl1['net_profit'] + pnl2['net_profit']

print(f"Truck 1 ({truck1['truck_name']}):  ${pnl1['net_profit']:,.2f}")
print(f"Truck 2 ({truck2['truck_name']}):  ${pnl2['net_profit']:,.2f}")
print(f'Combined net profit:   ${combined_profit:,.2f}')


## Section 4 — Validation

Add at least **5 assert statements** validating key constraints and math.
Your notebook should be able to run top-to-bottom with all assertions passing.


In [ ]:
# ── Validation assertions ──────────────────────────────────────────────────────
# TODO: add at least 5 assertions. Starter examples below — keep these and add more.

# 1. No truck carries more than its physical capacity
# for result in single_results:
#     truck = next(t for t in truck_list if t['truck_name'] == result['truck'])
#     assert result['entrees'] <= truck['capacity'], \
#         f"{result['truck']} over capacity: {result['entrees']} > {truck['capacity']}"
# print('✅ All trucks within capacity')

# 2. No single-truck scenario exceeds the daily supply cap
# for result in single_results:
#     assert result['entrees'] <= DAILY_SUPPLY_CAP, \
#         f"{result['truck']} exceeded supply cap: {result['entrees']} > {DAILY_SUPPLY_CAP}"
# print('✅ All scenarios respect daily supply cap')

# 3. TODO: validate gross profit formula for best single-truck scenario

# 4. TODO: validate two-truck combined cap if you deployed two trucks

# 5. TODO: validate that no restaurant appears in both truck splits (two-truck only)

print('Add your assertions here.')


## Our Recommendation

*Replace this cell with your group's written recommendation.*

Answer these questions:
1. Which truck (or truck pair) do you recommend for today's supply level, and why?
2. What net profit does your recommended configuration produce?
3. Which restaurants did you choose to serve, and which did you skip? Why?
4. What trade-offs did you encounter? (e.g., a high-margin restaurant that was
   too far out of the way to justify serving)
5. How does your recommendation change at different supply levels?
   At what supply level (if any) does a different truck or configuration win?
6. How did your group approach the HQ return leg in your routing? Did it materially
   affect your truck selection?


## Submission Checklist

Before submitting, confirm:

- [ ] Notebook runs top-to-bottom without errors
- [ ] `DAILY_SUPPLY_CAP` is set as a single variable (never hardcoded downstream)
- [ ] HQ address is geocoded; all routes start and end at HQ
- [ ] All 5+ assert statements pass
- [ ] At least 3 scenarios evaluated at `DAILY_SUPPLY_CAP = 165` (shown in comparison table)
- [ ] At least one additional supply level tested (e.g., 100 or 120 entrees)
- [ ] Final P&L table for your recommended configuration
- [ ] `Our Recommendation` markdown cell is complete with specific numbers
- [ ] Notebook is committed and pushed to your group's GitHub repo
- [ ] GitHub repo link submitted on Canvas before start of X11D2

**File name:** `X11_Project1_[GroupName].ipynb`  
**Canvas assignment:** Project One — Group Challenge  
**Due:** X11D2 (Thursday), start of class  

> ⚡ On X11D2 your instructor will announce the actual supply available for that day.
> Be ready to update `DAILY_SUPPLY_CAP`, re-run, and present your result within minutes.
